# Formativa 1 — Framingham Heart Study

Grupo 3 — MCDI501

## Introducción

### El dataset
El *Framingham Heart Study* es un estudio cardiovascular de largo plazo que sigue a residentes de la ciudad de
Framingham (Massachusetts, EE. UU.) y es una de las fuentes más citadas sobre factores de riesgo de enfermedad
cardiovascular. Este trabajo usa la versión pública publicada en Kaggle (*Heart Disease Prediction using Logistic
Regression*), que contiene **4.238 participantes** y **16 variables**: 15 características medidas al inicio
del seguimiento y una variable objetivo.

### Variable objetivo
`TenYearCHD` indica si la persona desarrolló **enfermedad coronaria (CHD) en los diez años siguientes**
(1 = sí, 0 = no). Es una variable binaria, y su prevalencia en la muestra es de 15,2% (644 de 4.238).

### Qué se busca
En contextos de salud pública y atención primaria, anticiparse al riesgo cardiovascular permite priorizar
controles y prevención. Pero la decisión se toma bajo incertidumbre: solo se observa una muestra de personas y
las asociaciones con el riesgo son débiles a moderadas. Este informe examina qué tan bien se pueden describir los
datos, cuánta incertidumbre tienen las estimaciones y si las diferencias observadas entre grupos (por ejemplo, por
edad, sexo o antecedentes de hipertensión) son estadísticamente distinguibles del azar.

### Alcance de este cuaderno
Reproduce los resultados del informe de la Formativa 1, en el mismo orden:
(1) estadística descriptiva, (2) estimación puntual e intervalos de confianza al 95% y (3) pruebas de hipótesis.
Semilla fija: `42`.

In [ ]:
import sys
print(sys.executable)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import kagglehub, shutil
from pathlib import Path

path = kagglehub.dataset_download("dileep070/heart-disease-prediction-using-logistic-regression")
destino = Path("../../data/raw")
destino.mkdir(parents=True, exist_ok=True)
shutil.copy(Path(path) / "framingham.csv", destino / "framingham.csv")
print("Copiado en:", (destino / "framingham.csv").resolve())

In [ ]:
df = pd.read_csv("../../data/raw/framingham.csv")
print(df.shape)
df.info()
df.head()

### Semilla, variable objetivo y estilo de las figuras
Semilla fija `42`. Paleta común para todas las figuras del informe (colores UNAB).

In [ ]:
from scipy import stats

SEED = 42
np.random.seed(SEED)
TARGET = "TenYearCHD"

# Las figuras se guardan directo en redaccion/, de donde las toma informe.tex al compilar
# (esa carpeta se versiona en git; así no hay dos copias que puedan desincronizarse).
FIGURAS = Path("../../redaccion/formativa-1-eda-inferencia/figuras")
FIGURAS.mkdir(parents=True, exist_ok=True)

# Estilo común de las figuras (paleta UNAB)
NAVY, RED, GREY, BG = "#0d2741", "#ae0000", "#8c8578", "#f6f1ea"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200,
    "font.size": 10, "axes.facecolor": BG,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": "#ddd5c6", "axes.axisbelow": True,
})
print("Semilla:", SEED, "| Objetivo:", TARGET)

## 1. Estadística descriptiva (ID1.2)

### 1.1 Tipos de variables y medidas resumen
Una variable objetivo binaria (`TenYearCHD`), seis indicadores binarios, una variable ordinal, un conteo
y siete variables numéricas continuas o cuasi-continuas.

In [ ]:
tipos = pd.Series({
    "TenYearCHD": "Binaria (objetivo)",
    "male": "Binaria", "currentSmoker": "Binaria", "BPMeds": "Binaria",
    "prevalentStroke": "Binaria", "prevalentHyp": "Binaria", "diabetes": "Binaria",
    "education": "Ordinal (1-4)",
    "cigsPerDay": "Conteo",
    "age": "Cuasi-continua",
    "totChol": "Continua", "sysBP": "Continua", "diaBP": "Continua",
    "BMI": "Continua", "heartRate": "Continua", "glucose": "Continua",
}, name="Tipo")
tipos.to_frame()

**Lectura.** El conjunto tiene 16 variables: el objetivo binario (`TenYearCHD`), seis indicadores binarios
(sexo, fumador actual, medicación antihipertensiva, ACV previo, hipertensión prevalente y diabetes), una
variable ordinal (nivel educativo), un conteo (cigarrillos por día) y siete variables continuas. Predominan las binarias, por lo que la media de esas variables se interpreta como proporción.

In [ ]:
# Tabla 1 del informe: media, mediana, desviación estándar, mínimo y máximo
cuant = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]
desc = df[cuant].describe().T[["mean", "50%", "std", "min", "max"]]
desc.columns = ["Media", "Mediana", "Desv. est.", "Mínimo", "Máximo"]
desc.round(2)

**Lectura.** La edad es aproximadamente simétrica (media 49,6 y mediana 49,0 años; rango de 32 a 70). El
colesterol total, la presión sistólica y la glucosa tienen la media por sobre la mediana (236,7 vs 234,0;
132,4 vs 128,0; 82,0 vs 78,0) y máximos muy alejados del resto (696 mg/dL, 295 mmHg y 394 mg/dL), señal de
asimetría a la derecha y valores extremos. En `cigsPerDay` la mediana es 0 y la media 9,0: más de la mitad
de los participantes no fuma y la media la elevan los fumadores intensos (hasta 70 cigarrillos diarios).


### 1.2 Calidad de datos
Desbalance del objetivo, datos faltantes, duplicados y valores extremos.

In [ ]:
n = len(df)
print(f"Prevalencia de CHD a 10 años : {df[TARGET].mean()*100:.2f}%  ({df[TARGET].sum():,} de {n:,})")
print(f"Filas duplicadas             : {df.duplicated().sum():,}")
print()
falt = pd.DataFrame({"n faltantes": df.isna().sum(), "% faltante": (df.isna().mean() * 100).round(2)})
print(falt[falt["n faltantes"] > 0].sort_values("n faltantes", ascending=False))
print(f"\nFilas con al menos un faltante: {df.isna().any(axis=1).sum():,} ({df.isna().any(axis=1).mean()*100:.1f}%)")

# Valores extremos (regla de Tukey: fuera de Q1-1.5*IQR, Q3+1.5*IQR)
print("\nValores extremos según Tukey:")
for c in ["totChol", "sysBP", "diaBP", "BMI", "glucose"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    iqr = q3 - q1
    k = ((df[c] < q1 - 1.5 * iqr) | (df[c] > q3 + 1.5 * iqr)).sum()
    print(f"  {c:10s}: {k:4d} ({k/df[c].notna().sum()*100:.1f}%)")

**Lectura.** El 15,2% de los participantes (644 de 4.238) desarrolló enfermedad coronaria en diez años, es
decir, unos 5,6 sanos por cada caso: el objetivo está desbalanceado. No hay filas duplicadas. Hay datos
faltantes en siete variables; la glucosa concentra la mayor parte (388 registros, 9,2%) y 582 filas (13,7%)
tienen al menos un faltante, que es lo que se perdería al eliminar filas incompletas. Los valores extremos
según la regla de Tukey van de 1,3% (colesterol) a 4,9% (glucosa) de las observaciones.

### Figura 1 - Variable objetivo y datos faltantes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) distribución de la variable objetivo
vc = df[TARGET].value_counts().sort_index()
labels = ["Sin CHD (0)", "Con CHD (1)"]
ax[0].barh(labels, vc.values, color=[NAVY, RED], height=0.5)
ax[0].invert_yaxis()
for i, v in enumerate(vc.values):
    ax[0].text(v + 50, i, f"{v:,}  ({v/n*100:.1f}%)", va="center", fontsize=9)
ax[0].set_xlim(0, vc.max() * 1.25)
ax[0].set_xlabel("Frecuencia")
ax[0].set_title("(a) Distribución de la variable objetivo")
ax[0].grid(axis="y", visible=False)

# (b) porcentaje de faltantes por variable
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
y = np.arange(len(miss))
ax[1].hlines(y, 0, miss.values, color="#bdb3a3", lw=2.5)
ax[1].scatter(miss.values, y, s=90, color=RED, edgecolor="#333333", linewidth=0.6, zorder=3)
for i, v in enumerate(miss.values):
    ax[1].text(v + 0.2, i, f"{v:.1f}%", va="center", fontsize=9)
ax[1].set_yticks(y)
ax[1].set_yticklabels(miss.index, fontsize=9)
ax[1].invert_yaxis()
ax[1].set_xlim(0, miss.max() * 1.25)
ax[1].set_xlabel("% faltante")
ax[1].set_title("(b) Datos faltantes por variable")
ax[1].grid(axis="y", visible=False)

plt.tight_layout()
plt.savefig(FIGURAS / "fig_panorama.png", bbox_inches="tight")
plt.show()

**Lectura.** El panel (a) muestra el desbalance de la variable objetivo: la exactitud global sería engañosa
(predecir siempre "sin CHD" acertaría 84,8%), por lo que habrá que evaluar con métricas de la clase
minoritaria. El panel (b) muestra que los faltantes son pocos salvo en la glucosa (9,2%) y la educación
(2,5%); el resto no supera el 1,3%.

### Figura 2 - Distribución de tres variables representativas
El colesterol total se recorta al percentil 99 solo para visualizarlo; los cálculos usan todos los datos.

In [ ]:
chol = df["totChol"].dropna()
sbp = df["sysBP"]
paneles = [
    (df["age"], "Edad", "años", "media", df["age"].mean()),
    (chol.clip(upper=chol.quantile(0.99)), "Colesterol total (≤P99)", "mg/dL", "mediana", chol.median()),
    (sbp, "Presión sistólica", "mmHg", "mediana", sbp.median()),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, (x, titulo, xlab, nombre, ref) in zip(axes, paneles):
    ax.hist(x, bins=40, histtype="stepfilled", color=NAVY, alpha=0.3, edgecolor=NAVY, linewidth=1.2)
    ax.axvline(ref, color=RED, ls="--", lw=1.8)
    ax.text(ref, ax.get_ylim()[1] * 0.93, f" {nombre}={ref:,.0f}", color=RED, fontsize=9, va="center")
    ax.set_title(titulo)
    ax.set_xlabel(xlab)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("Frecuencia")

plt.tight_layout()
plt.savefig(FIGURAS / "fig_distribuciones.png", bbox_inches="tight")
plt.show()

**Lectura.** La edad se distribuye de forma aproximadamente simétrica en torno a 50 años. El colesterol total
y la presión sistólica son asimétricos a la derecha: la mediana (234 mg/dL y 128 mmHg) queda bajo la media y hay
una cola de valores altos. Por eso en esas dos variables se usa la mediana como referencia, y el colesterol se
recorta al percentil 99 solo para visualizarlo.

### Figura 3 - Matriz de correlaciones (triángulo inferior)

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

corr = df.drop(columns=TARGET).corr()
k = len(corr)
mask = np.triu(np.ones((k, k), dtype=bool), k=1)   # oculta el triángulo superior
cmap = LinearSegmentedColormap.from_list("div", [RED, "#ede4d6", NAVY])

fig, ax = plt.subplots(figsize=(8, 7))
ax.set_facecolor("white")
ax.grid(False)
im = ax.pcolormesh(np.ma.masked_array(corr.values, mask), cmap=cmap, vmin=-1, vmax=1,
                   edgecolors="white", linewidth=1.5)
ax.invert_yaxis()
for i in range(k):
    for j in range(i + 1):
        v = corr.values[i, j]
        ax.text(j + 0.5, i + 0.5, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(v) > 0.5 else "#333333")
ax.set_xticks(np.arange(k) + 0.5)
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7.5)
ax.set_yticks(np.arange(k) + 0.5)
ax.set_yticklabels(corr.columns, fontsize=7.5)
ax.set_title("Matriz de correlaciones entre variables")
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Correlación de Pearson")

plt.tight_layout()
plt.savefig(FIGURAS / "fig_correlacion.png", bbox_inches="tight")
plt.show()

# Pares con mayor |r| fuera de la diagonal
pares = corr.where(np.tril(np.ones((k, k), dtype=bool), k=-1)).stack()
pares.reindex(pares.abs().sort_values(ascending=False).index).head(5).round(3)

**Lectura.** Las correlaciones más altas son presión sistólica con diastólica (r = 0,78), fumador actual con
cigarrillos por día (r = 0,77, relación en parte estructural: quien no fuma tiene 0 cigarrillos), hipertensión
prevalente con presión sistólica (r = 0,70) y diabetes con glucosa (r = 0,62). El resto de las correlaciones
son débiles a moderadas.

In [ ]:
# Correlación de cada variable con el objetivo (se menciona en la interpretación del informe)
df.corr()[TARGET].drop(TARGET).sort_values().round(4)

## 2. Estimación puntual e intervalos de confianza (ID1.3)
Se estiman tres parámetros poblacionales con un 95% de confianza, usando la aproximación normal
que respalda el Teorema Central del Límite: $\bar{x} \pm z_{0.975} \cdot EE$, con $EE = s/\sqrt{n}$
para las medias y $EE = \sqrt{\hat{p}(1-\hat{p})/n}$ para la proporción.

In [ ]:
# Tabla 2 del informe: estimación puntual e IC 95% para tres parámetros poblacionales
z = stats.norm.ppf(0.975)

def ic_media(serie):
    x = serie.dropna()
    m, ee, ni = x.mean(), x.std(ddof=1) / np.sqrt(len(x)), len(x)
    return m, m - z * ee, m + z * ee, ni

def ic_proporcion(serie):
    p, ni = serie.mean(), len(serie)
    ee = np.sqrt(p * (1 - p) / ni)
    return p, p - z * ee, p + z * ee, ni

parametros = [
    ("Edad media (años)", df["age"], False),
    ("Colesterol total medio (mg/dL)", df["totChol"], False),
    ("Proporción de CHD a 10 años", df[TARGET], True),
]

filas = []
for nombre, serie, es_proporcion in parametros:
    est, lo, hi, ni = ic_proporcion(serie) if es_proporcion else ic_media(serie)
    filas.append({
        "Parámetro": nombre,
        "Estimación": round(est * 100, 2) if es_proporcion else round(est, 2),
        "IC 95% inf.": round(lo * 100, 2) if es_proporcion else round(lo, 2),
        "IC 95% sup.": round(hi * 100, 2) if es_proporcion else round(hi, 2),
        "n": ni,
    })

tabla_ic = pd.DataFrame(filas)
tabla_ic

**Lectura.** Con 4.238 participantes, el Teorema Central del Límite respalda la aproximación normal en
los tres casos, y los intervalos salen angostos por el tamaño de la muestra. Con un 95% de confianza,
la edad media poblacional está entre 49,33 y 49,84 años, y la proporción de personas que desarrollan
CHD a 10 años cae entre 14,12% y 16,28%. El colesterol total medio (236,72 mg/dL) se estimó con 50
observaciones menos por datos faltantes (n = 4.188), pero como representan solo 1,2% del total el
intervalo [235,37; 238,07] sigue siendo confiable. Como vimos en la Sección 1, el colesterol es
asimétrico a la derecha; el TCL garantiza que la *media muestral* se distribuya de forma aproximadamente
normal pese a eso, así que el intervalo es válido, pero conviene recordar que la mediana (234,0 mg/dL)
sigue siendo la referencia más representativa del participante típico.

## 3. Pruebas de hipótesis (ID1.4)
Nivel de significancia: $\alpha = 0{,}05$.

### 3.1 Prueba *t* de Welch: edad entre quienes desarrollan CHD y quienes no
$H_0:\ \mu_{\text{CHD}} = \mu_{\text{no CHD}}$ vs. $H_1:\ \mu_{\text{CHD}} \neq \mu_{\text{no CHD}}$.
Se usa Welch (no asume varianzas iguales) porque los grupos tienen tamaños muy distintos.

In [ ]:
edad_chd = df.loc[df[TARGET] == 1, "age"]
edad_no = df.loc[df[TARGET] == 0, "age"]
print(f"Varianzas: con CHD = {edad_chd.var():.1f}, sin CHD = {edad_no.var():.1f}")

t, p = stats.ttest_ind(edad_chd, edad_no, equal_var=False)
n1, n0 = len(edad_chd), len(edad_no)
sp = np.sqrt(((n1 - 1) * edad_chd.var(ddof=1) + (n0 - 1) * edad_no.var(ddof=1)) / (n1 + n0 - 2))
d = (edad_chd.mean() - edad_no.mean()) / sp

print(f"Media con CHD : {edad_chd.mean():.2f} años (n={n1:,})")
print(f"Media sin CHD : {edad_no.mean():.2f} años (n={n0:,})")
print(f"t de Welch = {t:.2f}   valor p {'< 0.0001' if p < 1e-4 else f'= {p:.4f}'}   d de Cohen = {d:.3f}")
print("Decisión:", "se rechaza H0" if p < 0.05 else "no se rechaza H0")

**Lectura.** La edad media es 54,15 años entre quienes desarrollan CHD (n=644) y 48,77 años entre
quienes no (n=3.594), una diferencia de casi 5,4 años. El estadístico es t = 15,58 con un valor
p < 0,0001, por lo que **se rechaza H0**: la diferencia es estadísticamente significativa. El
tamaño del efecto (d de Cohen = 0,64) es moderado a grande — la edad separa razonablemente bien
a ambos grupos, consistente con que fue la variable con mayor correlación con el objetivo
(r = 0,23) en la Sección 1.

### 3.2 Prueba $\chi^2$ de independencia: hipertensión prevalente y CHD
$H_0:$ hipertensión prevalente y CHD son independientes vs. $H_1:$ están asociadas.

In [ ]:
tabla = pd.crosstab(df["prevalentHyp"], df[TARGET])
chi2, p_chi, gl, esperadas = stats.chi2_contingency(tabla)
V = np.sqrt(chi2 / (tabla.values.sum() * (min(tabla.shape) - 1)))

print(f"Chi² = {chi2:,.1f}   gl = {gl}   valor p {'< 0.0001' if p_chi < 1e-4 else f'= {p_chi:.4f}'}   V de Cramér = {V:.3f}")
print(f"Frecuencia esperada mínima: {esperadas.min():.0f} (supuesto >= 5 se cumple)")
print("Decisión:", "se rechaza H0" if p_chi < 0.05 else "no se rechaza H0")

tasa_hiper = df.groupby("prevalentHyp")[TARGET].mean() * 100
tasa_hiper.index = ["Sin hipertensión", "Con hipertensión"]
tasa_hiper.round(1).to_frame("% con CHD a 10 años")

### Figura 4 - Tasa de CHD según hipertensión prevalente

In [ ]:
media_global = df[TARGET].mean() * 100
x = np.arange(2)

fig, ax = plt.subplots(figsize=(6, 4.2))
ax.bar(x, tasa_hiper.values, color=[NAVY, RED], width=0.5)
ax.axhline(media_global, color=GREY, ls="--", lw=1.5)
ax.text(1.3, media_global, f" media global = {media_global:.1f}%", color=GREY, fontsize=9, va="center")
for i, v in enumerate(tasa_hiper.values):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(tasa_hiper.index)
ax.set_ylabel("% con CHD a 10 años")
ax.set_title("Tasa de CHD según hipertensión prevalente")
ax.set_xlim(-0.6, 1.9)
ax.grid(axis="x", visible=False)

plt.tight_layout()
plt.savefig(FIGURAS / "fig_hipertension.png", bbox_inches="tight")
plt.show()

**Lectura.** La asociación es significativa ($\chi^2$ = 132,6; gl = 1; p < 0,0001), con una
intensidad débil a moderada (V de Cramér = 0,18). La frecuencia esperada mínima (200) supera
ampliamente el umbral de 5, así que el supuesto de la prueba se cumple. En términos prácticos:
el 24,7% de quienes tienen hipertensión prevalente desarrolla CHD a 10 años, frente al 10,9% de
quienes no la tienen — un **riesgo relativo de 2,3 veces**. Es una señal clínicamente relevante,
aunque menos marcada que la que separa a los grupos por edad.

## 4. Síntesis

- **Descriptiva:** objetivo desbalanceado (15,2% con CHD), faltantes concentrados en glucosa (9,2%)
  y educación (2,5%), sin duplicados ni valores imposibles. La edad y la presión sistólica son las
  variables más correlacionadas con el objetivo (r = 0,23 y 0,22).
- **Estimación:** intervalos de confianza estrechos por el tamaño muestral (edad media 49,58
  [49,33; 49,84]; proporción de CHD 15,20% [14,12%; 16,28%]).
- **Pruebas:** la edad difiere con fuerza entre grupos (d = 0,64), y la hipertensión prevalente
  duplica el riesgo de CHD a 10 años (riesgo relativo de 2,3).

En conjunto, estos resultados apuntan a que, en un contexto de atención primaria, controlar la
presión arterial y priorizar el seguimiento de pacientes de mayor edad podría anticipar casos de
riesgo cardiovascular antes de que la enfermedad coronaria se manifieste.